# VoltIQ – 03: Data Cleaning

**Objective**: Apply in-memory cleaning pipelines to all datasets.

Rules:
- Original CSV files are **never modified**.
- All cleaning occurs in memory.
- Outliers are **flagged** (not removed).
- Every step is logged.

## 1. Setup & Imports

In [ ]:
import sys, os, logging
project_root = os.path.abspath(os.path.join(os.getcwd(), '..'))
if project_root not in sys.path:
    sys.path.insert(0, project_root)

logging.basicConfig(level=logging.INFO,
    format='[%(levelname)s] %(name)s — %(message)s')

from app.utils.data_loader import DataLoader
from app.utils.cleaning import DataCleaner

loader = DataLoader()
raw_dfs = loader.load_all()
cleaner = DataCleaner()
print('Raw datasets loaded:', list(raw_dfs.keys()))

## 2. Clean All Datasets

In [ ]:
clean_dfs = cleaner.clean_all(raw_dfs)
print(f'\nCleaned {len(clean_dfs)} datasets.')

## 3. Compare Raw vs Clean Shapes

In [ ]:
import pandas as pd

comparison = []
for key in raw_dfs:
    if key in clean_dfs:
        comparison.append({
            'Dataset':     key,
            'Raw Rows':    len(raw_dfs[key]),
            'Clean Rows':  len(clean_dfs[key]),
            'Raw Cols':    len(raw_dfs[key].columns),
            'Clean Cols':  len(clean_dfs[key].columns),
            'Rows Removed': len(raw_dfs[key]) - len(clean_dfs[key])
        })

cmp_df = pd.DataFrame(comparison).set_index('Dataset')
print(cmp_df.to_string())

## 4. Inspect Outlier Flags (Battery)

In [ ]:
battery_clean = clean_dfs['battery']
outlier_cols = [c for c in battery_clean.columns if c.endswith('_outlier')]
print('Outlier flag columns:', outlier_cols)
for col in outlier_cols:
    n_flagged = battery_clean[col].sum()
    print(f'  {col}: {n_flagged:,} outliers flagged')

## 5. Verify Coordinate Validation (Charging Stations)

In [ ]:
stations_clean = clean_dfs['charging_stations']
if 'invalid_coordinates' in stations_clean.columns:
    n_invalid = stations_clean['invalid_coordinates'].sum()
    print(f'Invalid coordinates flagged: {n_invalid:,}')
    print(stations_clean[stations_clean['invalid_coordinates']].head(3))
else:
    print('No invalid coordinates found.')

## 6. Inspect Cleaned Battery Dataset

In [ ]:
print('Battery Clean Shape:', battery_clean.shape)
print('\nNull counts after cleaning:')
print(battery_clean.isnull().sum()[battery_clean.isnull().sum() > 0])
print('\nSample rows:')
print(battery_clean.head(3).to_string())

## 7. Inspect Cleaned Fleet Readiness Dataset

In [ ]:
fleet_clean = clean_dfs['fleet_readiness']
print('Fleet Clean Shape:', fleet_clean.shape)
print('\nEV_Readiness_Score range:', fleet_clean['EV_Readiness_Score'].min(),
      'to', fleet_clean['EV_Readiness_Score'].max())

## Summary

All datasets cleaned in memory. Duplicates removed, nulls imputed, types coerced, strings normalised, coordinates validated, and outliers flagged.

**Next step** → `04_Feature_Preparation.ipynb`